# Menu Detector

Classifies food photos (pilaf, kebab, ...) using transfer learning on a pretrained ResNet18.
Uses the dataset built by `data_scraping.ipynb` (`food101_dataset/<class>/*.jpg` on Google Drive).
Run in Colab with a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

DATASET_DIR = "/content/drive/MyDrive/food101_dataset"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

weights = models.ResNet18_Weights.DEFAULT
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), normalize,
])
val_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), normalize,
])

# Two views of the same folder so train and val get different transforms
train_full = datasets.ImageFolder(DATASET_DIR, transform=train_tf)
val_full = datasets.ImageFolder(DATASET_DIR, transform=val_tf)
classes = train_full.classes

idx = torch.randperm(len(train_full)).tolist()
split = int(0.8 * len(idx))
train_ds, val_ds = Subset(train_full, idx[:split]), Subset(val_full, idx[split:])

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=32, num_workers=2)
print(f"classes={classes}  train={len(train_ds)}  val={len(val_ds)}")

In [ ]:
model = models.resnet18(weights=weights)
for p in model.parameters():
    p.requires_grad = False          # freeze the pretrained backbone
model.fc = nn.Linear(model.fc.in_features, len(classes))  # new trainable head
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

In [ ]:
def run_epoch(dl, train):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n

best_acc = 0.0
for epoch in range(10):
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    va_loss, va_acc = run_epoch(val_dl, train=False)
    print(f"epoch {epoch+1:2d}  train loss {tr_loss:.3f} acc {tr_acc:.2%}  |  val loss {va_loss:.3f} acc {va_acc:.2%}")
    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({"state_dict": model.state_dict(), "classes": classes},
                   f"{DATASET_DIR}/menu_detector.pt")
print(f"best val acc: {best_acc:.2%}")

## Predict a single image

In [ ]:
from PIL import Image

def predict(path):
    ckpt = torch.load(f"{DATASET_DIR}/menu_detector.pt", map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    x = val_tf(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = model(x).softmax(1)[0]
    for c, p in sorted(zip(ckpt["classes"], probs.tolist()), key=lambda t: -t[1]):
        print(f"{c:10s} {p:.2%}")

# predict("/content/my_plate.jpg")